# Get labelized signals from Lepreau pig farm

To extract a labeled signal from the database, you need to select three parameters:
* the label identifier (integer);
* the file identifier (integer) among all the files containing these labeled signals ;
* the file section identifier (integer) among all file sections that may contain this label.

In [192]:
import numpy as np
from IPython import display
from megamicros_aidb.query.db import AidbSession
from megamicros.data import MuAudio
from megamicros.log import log
from megamicros.core_base import MU_MEMS_SENSIBILITY

log.setLevel( "INFO" )

FRAME_LENGTH = 1024
SAMPLING_FREQUENCY = 10000
CUTOFF_FREQUENCY = 2500

LABEL_SOW_FEEDING_CALL = 18
LABEL_PIGLET_SQUEALS = 15
LABEL_SOW_GRUNT_NERVOUS = 16
LABEL_ROOM_NOISE = 29
LABEL_SOW_GRUNT = 8
LABEL_SOW_GRUNT_MODSTRESS  = 1
LABEL_SOW_SCREAMS = 3
LABEL_PIGLET_SQUEALS_2 = 5

# choose your label:
LABEL_ID = LABEL_SOW_GRUNT_NERVOUS

In [193]:
# Choose your file
with AidbSession(
    dbhost='http://dbwelfare.biimea.io/',
    login='ailab',
    email='bruno.gas@biimea.com',
    password='#T;uZnQ5UJ_JC~&' ) as session:
        label_id = LABEL_ID
        labelings_file = session.load_labelings( label_id=label_id )
        print( f"Fichiers trouvés:" )
        for labeling_file in labelings_file:
            print( f" > [{labeling_file['sourcefile_id']}: {labeling_file['sourcefile_filename']}]")
        sourcefile_id = int( input( 'Numéro identificateur du fichier à sélectionner:' ) )
        limit = 100
        channels = list( np.arange( 32 ) + 1 )
        signals = session.load_labelized( sourcefile_id=sourcefile_id, label_id=label_id, limit=limit, channels=channels )

2023-08-29 22:57:17,715 [INFO]:  .Try connecting on endpoint database http://dbwelfare.biimea.io/...
2023-08-29 22:57:18,109 [INFO]:  .Got HTTP 200 status code from server
2023-08-29 22:57:18,110 [INFO]:  .Received CSRF token: 2m6sWNuLevdcv2bkkYpFqiqlqLunL6Ah. Update session with
2023-08-29 22:57:18,111 [INFO]:  .Received session id: cj3ftmyqkpvtijsliarg5vuexm5wawtp
2023-08-29 22:57:18,111 [INFO]:  .Successfully connected on http://dbwelfare.biimea.io/
2023-08-29 22:57:18,112 [INFO]:  .Downloading labelings from http://dbwelfare.biimea.io/...
2023-08-29 22:57:18,157 [INFO]:  .Received 6 labelings


Fichiers trouvés:
 > [7839: mu5h-20220815-094753.h5]
 > [7839: mu5h-20220815-094753.h5]
 > [5838: mu5h-20220720-131642.h5]
 > [7135: mu5h-20220722-094517.h5]
 > [7135: mu5h-20220722-094517.h5]
 > [7135: mu5h-20220722-094517.h5]


Numéro identificateur du fichier à sélectionner: 7135


2023-08-29 22:57:26,833 [INFO]:  .Downloading labelized audio files from http://dbwelfare.biimea.io/...
2023-08-29 22:57:26,875 [INFO]:  .Found 3 labelized audio files
2023-08-29 22:57:26,877 [INFO]:  .Limit is set to 100 audio files
2023-08-29 22:57:26,878 [INFO]:  .Downloading metadata for object 'sourcefile' [7135]...
2023-08-29 22:57:26,986 [INFO]:  Object sourcefile found with identifier [7135] 
2023-08-29 22:57:27,335 [INFO]:  .Downloading metadata for object 'sourcefile' [7135]...
2023-08-29 22:57:27,476 [INFO]:  Object sourcefile found with identifier [7135] 


I'm a NDarray signal with frame size = 5704 and frame number = 1


2023-08-29 22:57:28,196 [INFO]:  .Downloading metadata for object 'sourcefile' [7135]...
2023-08-29 22:57:28,299 [INFO]:  Object sourcefile found with identifier [7135] 


I'm a NDarray signal with frame size = 11506 and frame number = 1


2023-08-29 22:57:28,602 [INFO]:  .Trying to disconnect from database http://dbwelfare.biimea.io/...
2023-08-29 22:57:28,647 [INFO]:  .Logout successful.


I'm a NDarray signal with frame size = 7099 and frame number = 1


In [194]:
# Choose your file section
print( f"{len(signals)} section audio récupérées: " )
for idx, aud in enumerate( signals ):
    print( f"Audio[{idx}]: {aud} -> label={aud.label}, channels number: {aud.channels_number} ({aud.samples_number} samples)")
selected = int( input( f"Choisir un signal (0 à {len(signals)-1}): " ) )
signal: MuAudio = signals[selected]

3 section audio récupérées: 
Audio[0]: 32 X 5704 audio signals (sf=10000.0 Hz) -> label=sow-grunt-nervous, channels number: 32 (5704 samples)
Audio[1]: 32 X 11506 audio signals (sf=10000.0 Hz) -> label=sow-grunt-nervous, channels number: 32 (11506 samples)
Audio[2]: 32 X 7099 audio signals (sf=10000.0 Hz) -> label=sow-grunt-nervous, channels number: 32 (7099 samples)


Choisir un signal (0 à 2):  1


In [195]:
# get infos
print( f"label={signal.label}" )
print( f"channels_number={signal.channels_number}" )
print( f"samples_number={signal.samples_number}" )
print( f"sampling_frequency={signal.sampling_frequency}" )

# Play sound using channel 0 and 1
left = np.array( signal.channel(0) )
right = np.array( signal.channel(1) )
sound = np.array( [left, right] )

label=sow-grunt-nervous
channels_number=32
samples_number=11506
sampling_frequency=10000.0


In [196]:
display.Audio(sound, rate=SAMPLING_FREQUENCY )

In [197]:
# Get the whole 32 channels signal as a numpy.ndarray
my_signal = signal()
print( np.shape(my_signal) )

(32, 11506)


## Location
See the autocalibrating of the square antenna with microphones from brown (0, (left,front)) to 31)
![image](antenna-square.png)

Pairs:
* MEMs [0 (front), 7 (back)] or [16 (back), 23(front)]
* MEMs [8 (left), 15 (right)] or [24 (right), 31(left)]

In [198]:
signal_front_back = np.array( [signal.channel(0), signal.channel(7)] )
signal_back_front = np.array( [signal.channel(16), signal.channel(23)] )
signal_left_right = np.array( [signal.channel(8), signal.channel(15)] )
signal_right_left = np.array( [signal.channel(24), signal.channel(31)] )

In [199]:
display.Audio(signal_front_back, rate=SAMPLING_FREQUENCY )

In [200]:
display.Audio(signal_back_front, rate=SAMPLING_FREQUENCY )

In [201]:
display.Audio(signal_left_right, rate=SAMPLING_FREQUENCY )

In [202]:
display.Audio(signal_right_left, rate=SAMPLING_FREQUENCY )

## Non working MEMS

In [203]:
display.Audio([signal.channel(26), signal.channel(27)], rate=SAMPLING_FREQUENCY )

In [204]:
display.Audio([signal.channel(2), signal.channel(3)], rate=SAMPLING_FREQUENCY )

In [205]:
display.Audio([signal.channel(14), signal.channel(20)], rate=SAMPLING_FREQUENCY )

In [206]:
display.Audio([signal.channel(21), signal.channel(21)], rate=SAMPLING_FREQUENCY )